In [ ]:
# Import all necessary packages
import torch
from torchmetrics.functional.regression import mean_absolute_error, pearson_corrcoef
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import distinctipy
import matplotlib.colors as mcolors
from plottable import ColumnDefinition, Table
import rootutils

# Set directories and load data
path_root = str(rootutils.find_root(indicator=".project-root"))
path_plots = f"{path_root}/plots"
path_data = f"{path_root}/data/age-regression"
df_feats = pd.read_excel(f"{path_data}/features.xlsx", index_col=0)
imms = df_feats.index.to_list()
df = pd.read_excel(f"{path_data}/data.xlsx")

gse_controls_count = df.loc[df['Status'] == 'Control', 'GSE'].value_counts()
gses_controls = gse_controls_count.index.values
gse_controls_ids = {gse: df.index[(df['Status'] == 'Control') & (df['GSE'] == gse)].values for gse in gses_controls}

# Prepare colors
def make_rgb_transparent(rgb, bg_rgb, alpha):
    return [alpha * c1 + (1 - alpha) * c2 for (c1, c2) in zip(rgb, bg_rgb)]

# Colors for GSEs
colors = distinctipy.get_colors(len(gses_controls), [mcolors.hex2color(mcolors.CSS4_COLORS['white']), mcolors.hex2color(mcolors.CSS4_COLORS['black'])], rng=1337)
colors_gse_controls = {gses_controls[gse_id]: colors[gse_id] for gse_id in range(len(gses_controls))}

# Plot Supplementary Figure S2
n_rows = 6 * 4
n_cols = 12
fig_height = 32
fig_width = 46
sns.set_theme(style='ticks')
fig, axs = plt.subplots(n_rows, n_cols, figsize=(fig_width, fig_height), height_ratios=[0.2, 0.2, 0.8, 0.15] * 6, gridspec_kw={'wspace':0.25, 'hspace': 0.05}, sharey=False, sharex=False)

for gse_id, (gse, gse_samples) in tqdm(enumerate(gse_controls_ids.items())):
    color_gse = colors_gse_controls[gse]
    data_gse = df.loc[gse_samples, ['GSE', 'Age', 'Status', 'Split', 'EpInflammAge']]
    data_gse['Error'] = data_gse['EpInflammAge'] - data_gse['Age']
    row_id, col_id = divmod(gse_id, n_cols)
    row_id_table = row_id * 4
    row_id_hist = row_id * 4 + 1
    row_id_scatter = row_id * 4 + 2
    row_id_empty = row_id * 4 + 3

    df_table = pd.DataFrame(index=['MAE', "Pearson's R", "Bias"], columns=['Train', 'Validation', 'Test', 'Total'])
    for part in ['Train', 'Validation', 'Test', 'Total']:
        
        if part != 'Total':
            pred = data_gse.loc[data_gse['Split'] == part, 'EpInflammAge'].values
            real = data_gse.loc[data_gse['Split'] == part, 'Age'].values
            errs = data_gse.loc[data_gse['Split'] == part, 'Error'].values
        else:
            pred = data_gse['EpInflammAge'].values
            real = data_gse['Age'].values
            errs = data_gse['Error'].values
            
        df_table.at['MAE', part] = f"{mean_absolute_error(torch.from_numpy(pred), torch.from_numpy(real)).numpy().item():0.3f}"
        df_table.at["Pearson's R", part] = f"{pearson_corrcoef(torch.from_numpy(pred), torch.from_numpy(real)).numpy().item():0.3f}"
        df_table.at["Bias", part] = f"{np.mean(errs):0.3f}"

    col_defs = [
        ColumnDefinition(
            name="index",
            title=gse if gse != 'GSEUNN' else 'This work',
            textprops={"ha": "center", "weight": "bold"},
            width=2.5,
        ),
        ColumnDefinition(
            name="Train",
            textprops={"ha": "left"},
            width=1.5,
            border="left"
        ),
        ColumnDefinition(
            name="Validation",
            textprops={"ha": "left"},
            width=2.2,
        ),
        ColumnDefinition(
            name="Test",
            textprops={"ha": "left"},
            width=1.5,
        ),
        ColumnDefinition(
            name="Total",
            textprops={"ha": "left"},
            width=1.5,
        )
    ]

    table = Table(
        df_table,
        column_definitions=col_defs,
        row_dividers=True,
        footer_divider=False,
        ax=axs[row_id_table, col_id],
        textprops={"fontsize": 8},
        row_divider_kw={"linewidth": 1, "linestyle": (0, (1, 1))},
        col_label_divider_kw={"linewidth": 1, "linestyle": "-"},
        column_border_kw={"linewidth": 1, "linestyle": "-"},
    ).autoset_fontcolors(colnames=['Train', 'Validation', 'Test', 'Total'])

    hist_bins = np.linspace(0, 120, 13)
    histplot = sns.histplot(
        data=data_gse,
        bins=hist_bins,
        edgecolor='k',
        linewidth=1,
        x="Age",
        color=color_gse,
        ax=axs[row_id_hist, col_id]
    )
    axs[row_id_hist, col_id].set_xticks([])
    axs[row_id_hist, col_id].set_xlim(0, 115)
    if col_id == 0:
        axs[row_id_hist, col_id].set_ylabel("Count")
    else:
        axs[row_id_hist, col_id].set_ylabel("")

    kdeplot = sns.kdeplot(
        data=data_gse.loc[data_gse['Split'].isin(['Train', 'Validation']), :],
        x='Age',
        y='EpInflammAge',
        fill=True,
        cbar=False,
        color=make_rgb_transparent(color_gse, (1, 1, 1), 0.25),
        cut=0,
        legend=False,
        ax=axs[row_id_scatter, col_id]
    )
    scatter = sns.scatterplot(
        data=data_gse.loc[data_gse['Split'] == 'Test', :],
        x='Age',
        y="EpInflammAge",
        linewidth=0.5,
        alpha=0.8,
        edgecolor="k",
        s=35,
        color=color_gse,
        ax=axs[row_id_scatter, col_id],
    )
    axs[row_id_scatter, col_id].axline((0, 0), slope=1, color="black", linestyle=":")
    axs[row_id_scatter, col_id].set_xlim(0, 115)
    axs[row_id_scatter, col_id].set_ylim(0, 115)
    if col_id == 0:
        axs[row_id_scatter, col_id].set_ylabel("Prediction")
    else:
        axs[row_id_scatter, col_id].set_ylabel("")
    if row_id_empty == n_rows - 1:
        axs[row_id_scatter, col_id].set_xlabel("Age")
    else:
        axs[row_id_scatter, col_id].set_xlabel("")
    axs[row_id_empty, col_id].axis('off')

fig.tight_layout()
fig.savefig(f"{path_plots}/supplementary-figure-s2.png", bbox_inches='tight', dpi=200)
fig.savefig(f"{path_plots}/supplementary-figure-s2.pdf", bbox_inches='tight')
plt.close(fig)